In [99]:
df = spark.table("silver.silver_workitems")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 101, Finished, Available, Finished, False)

In [100]:
display(df.limit(5))
print("Silver Count:", df.count())

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 102, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ab05de7-3bb6-4d31-99dd-01c5e0959a46)

Silver Count: 499


**Dim Product**

In [101]:
dim_product = df.select(
    "product_name",
    "product_id"
).distinct()\
.orderBy("product_name")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 103, Finished, Available, Finished, False)

In [102]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

ws = Window.orderBy("product_name")

dim_product = dim_product.withColumn(
    "product_key",
    row_number().over(ws)
)


StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 104, Finished, Available, Finished, False)

In [103]:
dim_product = dim_product.select(
    "product_key",
    "product_id",
    "product_name"
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 105, Finished, Available, Finished, False)

In [104]:
dim_product.write.mode("overwrite")\
    .saveAsTable("gold.dim_products")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 106, Finished, Available, Finished, False)

**Dim Date**

In [105]:
dim_date = df.select(
    "created_date",
    "reporting_month"
).distinct()

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 107, Finished, Available, Finished, False)

In [106]:
dim_date = dim_date.withColumn(
    "reporting_year",
    year(col("created_date"))
)
dim_date = dim_date.withColumn(
    "reporting_month_num", 
    month(col("created_date"))
)
dim_date = dim_date.withColumn(
    "month_name",
    date_format(col("created_date"), "MMMM")
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 108, Finished, Available, Finished, False)

In [107]:
ws = Window.orderBy("created_date")

dim_date = dim_date.withColumn(
    "date_key",
    row_number().over(ws)
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 109, Finished, Available, Finished, False)

In [108]:
dim_date = dim_date.select(
    "date_key",
    "created_date",
    "reporting_month",
    "reporting_year",
    "reporting_month_num",
    "month_name"
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 110, Finished, Available, Finished, False)

In [109]:
dim_date.write.mode("overwrite")\
    .saveAsTable("gold.dim_date")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 111, Finished, Available, Finished, False)

In [110]:
fact_df = df.filter(col("is_bug") == 1)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 112, Finished, Available, Finished, False)

In [111]:
fact_df = fact_df.alias("f")
dim_product = dim_product.alias("p")
dim_date = dim_date.alias("d")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 113, Finished, Available, Finished, False)

In [112]:
fact_df  = fact_df.join(
    dim_product,
    col("f.product_name") == col("p.product_name"),
    "left"
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 114, Finished, Available, Finished, False)

In [113]:
fact_df = fact_df.join(
    dim_date,
    (
        (col("f.created_date") == col("d.created_date")) &
        (col("f.reporting_month") == col("d.reporting_month"))
    ),
    "left"
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 115, Finished, Available, Finished, False)

**Fact Table**

In [114]:
fact_bug_metrics = fact_df.select(
    col("p.product_key"),
    col("p.product_name"),
    col("d.date_key"),
    col("f.work_item_id"),
    col("f.is_bug")
)

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 116, Finished, Available, Finished, False)

In [115]:
fact_bug_metrics.write.mode("overwrite")\
    .saveAsTable("gold.fact_bug_metrics")

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 117, Finished, Available, Finished, False)

In [116]:
display(fact_bug_metrics.limit(4))
print("Fact Count :", fact_bug_metrics.count())

StatementMeta(, 2557caa0-2d21-4f15-b047-50eff2bc7108, 118, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5d90e06-e062-4f98-a330-ba7fb67a98f5)

Fact Count : 30
